# Imports

In [12]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import torch
from torch import nn
from torch.nn import functional as F
import torch.optim as optim
from tqdm import tqdm

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [13]:
SEED = 4207
torch.manual_seed(SEED)
np.random.seed(SEED)

# Data Loading

In [14]:
train_data = datasets.MNIST(
    root = 'data',
    train = True,
    transform = ToTensor(),
    download = True,
)
test_data = datasets.MNIST(
    root = 'data',
    train = False,
    transform = ToTensor()
)

In [15]:
def drop_zero(ds):
    idx = (ds.targets != 0).nonzero(as_tuple=True)[0]
    X = ds.data[idx].float().div(255).unsqueeze(1)
    y = ds.targets[idx] - 1
    return X, y

X_all, y_all = drop_zero(train_data)
X_test, y_test = drop_zero(test_data)

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(X_all), generator=g)
n_val = int(len(X_all) * 10_000 / 60_000)
val_idx, tr_idx = perm[:n_val], perm[n_val:]
X_val, y_val = X_all[val_idx], y_all[val_idx]
X_tr,  y_tr  = X_all[tr_idx],  y_all[tr_idx]

loaders = {
    'train': DataLoader(TensorDataset(X_tr, y_tr), batch_size=100, shuffle=True),
    'val':   DataLoader(TensorDataset(X_val, y_val), batch_size=500),
    'test':  DataLoader(TensorDataset(X_test, y_test), batch_size=500),
}

# Model

In [16]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, 5)
        self.conv2 = nn.Conv2d(10, 20, 5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 9)
    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        return self.fc2(x)

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
model = CNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [19]:
def train(epoch):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    for batch_idx, (data, target) in enumerate(loaders['train']):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        total += target.size(0)

    train_loss = epoch_loss / len(loaders['train'])
    train_acc = 100. * correct / total
    print(f'Epoch {epoch} - Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.2f}%')
    return train_loss

In [20]:
def test():
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loaders['test']:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += loss_fn(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            total += target.size(0)

    test_loss /= len(loaders['test'])
    test_acc = 100. * correct / total
    print(f'Test set - Average Loss: {test_loss:.4f}, Accuracy: {correct}/{total} ({test_acc:.2f}%)\n')
    return test_loss, test_acc

In [21]:
for epoch in tqdm(range(1, 11), desc="Training Progress"):
  train(epoch)
test()

Training Progress:  10%|█         | 1/10 [00:01<00:13,  1.51s/it]

Epoch 1 - Train Loss: 0.7017, Train Accuracy: 77.17%


Training Progress:  20%|██        | 2/10 [00:03<00:12,  1.51s/it]

Epoch 2 - Train Loss: 0.3020, Train Accuracy: 91.06%


Training Progress:  30%|███       | 3/10 [00:04<00:10,  1.51s/it]

Epoch 3 - Train Loss: 0.2422, Train Accuracy: 92.80%


... (11 outputs left out)

Epoch 9 - Train Loss: 0.1497, Train Accuracy: 95.56%


Training Progress: 100%|██████████| 10/10 [00:16<00:00,  1.64s/it]

Epoch 10 - Train Loss: 0.1428, Train Accuracy: 95.79%
Test set - Average Loss: 0.0480, Accuracy: 8906/9020 (98.74%)



(0.048016818264793405, 98.73614190687361)

In [22]:
model.eval()

def get_probs(X):
    with torch.no_grad():
        out = []
        for i in range(0, len(X), 500):
            out.append(F.softmax(model(X[i:i+500].to(device)), dim=1).cpu())
        return torch.cat(out).numpy()

np.save('probs_test.npy', get_probs(X_test))
np.save('labels_test.npy', (y_test + 1).numpy())
np.save('probs_val.npy', get_probs(X_val))
np.save('labels_val.npy', (y_val + 1).numpy())

torch.save(model.state_dict(), 'cnn.pt')